In [3]:
import os
import sys

project_path = '/home/kunal/code/CDR_Meets_LLMs'
os.chdir(project_path)

In [6]:
import pandas as pd
import os
import json

# File paths
base_path = '/home/kunal/code/CDR_Meets_LLMs'
reviews_books_path = os.path.join(base_path, 'data', 'reviews_Books.jsonl')
reviews_movies_path = os.path.join(base_path, 'data', 'reviews_Movies_and_TV.jsonl')
meta_books_path = os.path.join(base_path, 'data', 'meta_Books.jsonl')
meta_movies_path = os.path.join(base_path, 'data', 'meta_Movies_and_TV.jsonl')

# Output folder
output_folder = os.path.join(base_path, 'dataset')
os.makedirs(output_folder, exist_ok=True)

# Corrected function to load reviews with proper column mapping
def load_reviews(path):
    data_list = []
    valid_records = 0
    total_lines = 0

    try:
        with open(path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                total_lines += 1
                try:
                    if line.strip():
                        record = json.loads(line)

                        # Map the actual column names to expected names
                        reviewer_id = record.get('user_id')  # user_id -> reviewerID
                        asin = record.get('parent_asin')     # parent_asin -> parent_asin
                        rating = record.get('rating')        # rating -> overall

                        if reviewer_id and asin and rating is not None:
                            filtered_record = {
                                'reviewerID': reviewer_id,
                                'parent_asin': asin,
                                'overall': rating
                            }
                            data_list.append(filtered_record)
                            valid_records += 1

                except json.JSONDecodeError as e:
                    if line_num % 100000 == 0:  # Print occasionally to show progress
                        print(f"Skipping malformed line {line_num + 1}: {e}")
                    continue

                # Progress indicator
                if line_num % 500000 == 0:
                    print(f"Processed {line_num} lines...")

    except FileNotFoundError as e:
        print(f"File not found: {e}")
        return pd.DataFrame()

    print(f"Completed: {total_lines} lines processed, {valid_records} valid records loaded")
    return pd.DataFrame(data_list)

# Load the data
print("Loading books data...")
books = load_reviews(reviews_books_path)
print(f"Books DataFrame shape: {books.shape}")

print("\nLoading movies data...")
movies = load_reviews(reviews_movies_path)
print(f"Movies DataFrame shape: {movies.shape}")

# 2) Find users who have reviews in both domains
print("\nFinding common users...")
common_users = set(books['reviewerID']).intersection(set(movies['reviewerID']))
print(f"Users in both domains: {len(common_users)}")

# 3) Filter each dataset to those users
books_filtered = books[books['reviewerID'].isin(common_users)]
movies_filtered = movies[movies['reviewerID'].isin(common_users)]
print(f"Books records for common users: {len(books_filtered)}")
print(f"Movies records for common users: {len(movies_filtered)}")

# 4) Keep only users with at least 5 reviews in each domain
min_reviews = 5
def filter_min_reviews(df, min_count):
    counts = df['reviewerID'].value_counts()
    valid_users = counts[counts >= min_count].index
    return df[df['reviewerID'].isin(valid_users)]

print(f"\nFiltering users with at least {min_reviews} reviews in each domain...")
books_filtered = filter_min_reviews(books_filtered, min_reviews)
movies_filtered = filter_min_reviews(movies_filtered, min_reviews)

# 5) Recompute common users after applying min_reviews filter
common_users = set(books_filtered['reviewerID']).intersection(set(movies_filtered['reviewerID']))
books_filtered = books_filtered[books_filtered['reviewerID'].isin(common_users)]
movies_filtered = movies_filtered[movies_filtered['reviewerID'].isin(common_users)]

print(f"Users with min {min_reviews} reviews in both domains: {len(common_users)}")
print(f"Final books records: {len(books_filtered)}")
print(f"Final movies records: {len(movies_filtered)}")

# 6) Sample a fraction of the remaining users (e.g., 10%)
sample_fraction = 0.10  # adjust fraction as needed
sample_users = pd.Series(list(common_users)).sample(frac=sample_fraction, random_state=42).tolist()

books_sample = books_filtered[books_filtered['reviewerID'].isin(sample_users)]
movies_sample = movies_filtered[movies_filtered['reviewerID'].isin(sample_users)]

print(f"\nSampled {sample_fraction*100}% of users: {len(sample_users)}")
print(f"Sampled books records: {len(books_sample)}")
print(f"Sampled movies records: {len(movies_sample)}")

# 7) Save the sampled review subsets
print("\nSaving review subsets...")
books_sample.to_json(os.path.join(output_folder, 'subset_reviews_Books.jsonl'),
                     orient='records', lines=True)
movies_sample.to_json(os.path.join(output_folder, 'subset_reviews_Movies_and_TV.jsonl'),
                      orient='records', lines=True)

# 8) Load metadata and save filtered versions
def load_meta(path):
    data_list = []
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                try:
                    if line.strip():
                        record = json.loads(line)
                        filtered_record = {
                            'parent_asin': record.get('parent_asin'),
                            'title': record.get('title')
                        }
                        if all(v is not None for v in filtered_record.values()):
                            data_list.append(filtered_record)
                except json.JSONDecodeError:
                    continue
    except FileNotFoundError as e:
        print(f"File not found: {e}")
        return pd.DataFrame()

    return pd.DataFrame(data_list)

print("Loading and filtering metadata...")
books_meta = load_meta(meta_books_path)
movies_meta = load_meta(meta_movies_path)

books_meta_filtered = books_meta[books_meta['parent_asin'].isin(books_sample['parent_asin'])]
movies_meta_filtered = movies_meta[movies_meta['parent_asin'].isin(movies_sample['parent_asin'])]

# 9) Save the filtered metadata subsets
books_meta_filtered.to_json(os.path.join(output_folder, 'subset_meta_Books.jsonl'),
                            orient='records', lines=True)
movies_meta_filtered.to_json(os.path.join(output_folder, 'subset_meta_Movies_and_TV.jsonl'),
                             orient='records', lines=True)

print("\n=== SUMMARY ===")
print("Subset creation complete. Files saved to:")
print(f"  {output_folder}/subset_reviews_Books.jsonl ({len(books_sample)} records)")
print(f"  {output_folder}/subset_reviews_Movies_and_TV.jsonl ({len(movies_sample)} records)")
print(f"  {output_folder}/subset_meta_Books.jsonl ({len(books_meta_filtered)} records)")
print(f"  {output_folder}/subset_meta_Movies_and_TV.jsonl ({len(movies_meta_filtered)} records)")
print(f"\nFinal dataset includes {len(sample_users)} users with reviews in both domains")

Loading books data...
Processed 0 lines...
Processed 500000 lines...
Processed 1000000 lines...
Processed 1500000 lines...
Processed 2000000 lines...
Processed 2500000 lines...
Processed 3000000 lines...
Processed 3500000 lines...
Processed 4000000 lines...
Processed 4500000 lines...
Processed 5000000 lines...
Processed 5500000 lines...
Processed 6000000 lines...
Processed 6500000 lines...
Processed 7000000 lines...
Processed 7500000 lines...
Processed 8000000 lines...
Processed 8500000 lines...
Processed 9000000 lines...
Processed 9500000 lines...
Processed 10000000 lines...
Processed 10500000 lines...
Processed 11000000 lines...
Processed 11500000 lines...
Processed 12000000 lines...
Processed 12500000 lines...
Processed 13000000 lines...
Processed 13500000 lines...
Processed 14000000 lines...
Processed 14500000 lines...
Processed 15000000 lines...
Processed 15500000 lines...
Processed 16000000 lines...
Processed 16500000 lines...
Processed 17000000 lines...
Processed 17500000 lines.

In [7]:
# Test the data loading function
from src.process_data import load_data

# Test with a small sample
books_data = load_data('Books', 'amazon')
movies_data = load_data('Movies_and_TV', 'amazon')

print("Books data shape:", books_data.shape)
print("Movies data shape:", movies_data.shape)
print("Books columns:", books_data.columns.tolist())

Books data shape: (356193, 4)
Movies data shape: (198922, 4)
Books columns: ['reviewerID', 'asin', 'overall', 'title']
